# Notebook 02 — Hardy Gate Demo

**Repo:** `unique-continuation-constraint-lab`  
**Role:** generate the reproducible Step 3 layer for the unique-continuation constraint pipeline.

Pipeline focus:

\[
\text{Hardy-type inequality}
\rightarrow
\text{concentration pays variation cost}
\rightarrow
\text{non-zero solutions cannot satisfy all constraints simultaneously}
\]

**same math · lifted clarity**

This refined version uses the shared repo modules:

```text
src/
  weights.py
  hardy_demo.py
  plotting.py
```

Generated figures:

```text
figures/
  step3_hardy_variation_cost_code.png
  notebook02_hardy_width_sweep_code.png
  notebook02_hardy_alignment_code.png
```


## 0. Setup

This notebook expects the repo layout:

```text
unique-continuation-constraint-lab/
  src/
  notebooks/
  figures/
```

It also includes a fallback path so it can run in Colab even if imports need path adjustment.


In [ ]:
import sys
from pathlib import Path

# Works from repo root, notebooks/, or Colab.
for candidate in [Path(".."), Path(".")]:
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate))
        break

import numpy as np
import matplotlib.pyplot as plt

try:
    from src.weights import gaussian, gradient, normalize
    from src.hardy_demo import concentration_measure, variation_cost, hardy_pair, width_sweep
    from src.plotting import set_style, ensure_fig_dir, save, add_caption, plot_alignment_bars
except Exception:
    # Minimal fallback if src is not available.
    def gaussian(x, center=0.0, width=1.0, amplitude=1.0):
        return amplitude * np.exp(-((x - center) ** 2) / (2 * width ** 2))

    def gradient(u, x):
        return np.gradient(u, x)

    def normalize(u):
        m = np.max(np.abs(u))
        return u if m == 0 else u / m

    def integrate(y, x):
        return np.trapz(y, x)

    def hardy_weight(x, eps=0.12):
        return 1.0 / (x**2 + eps)

    def concentration_measure(u, x, eps=0.12):
        return integrate(hardy_weight(x, eps=eps) * np.abs(u)**2, x)

    def variation_cost(u, x):
        return integrate(np.abs(gradient(u, x))**2, x)

    def hardy_pair(u, x, eps=0.12):
        return {"concentration": concentration_measure(u, x, eps=eps),
                "variation_cost": variation_cost(u, x)}

    def width_sweep(x, widths, eps=0.12, amplitude=1.0):
        concentration = []
        variation = []
        for width in widths:
            u = gaussian(x, width=width, amplitude=amplitude)
            concentration.append(concentration_measure(u, x, eps=eps))
            variation.append(variation_cost(u, x))
        return {"widths": np.asarray(widths),
                "concentration": np.asarray(concentration),
                "variation_cost": np.asarray(variation)}

    def set_style():
        plt.rcParams.update({
            "figure.figsize": (10, 4.8),
            "font.size": 12,
            "axes.grid": True,
            "grid.alpha": 0.25,
            "axes.spines.top": False,
            "axes.spines.right": False,
        })

    def ensure_fig_dir(path="figures"):
        candidates = [Path("../figures"), Path(path)]
        for c in candidates:
            if c.exists():
                c.mkdir(parents=True, exist_ok=True)
                return c
        candidates[0].mkdir(parents=True, exist_ok=True)
        return candidates[0]

    def save(fig, path, dpi=180):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=dpi, bbox_inches="tight")

    def add_caption(ax, text, y=-0.22):
        ax.text(0.02, y, text, transform=ax.transAxes,
                fontsize=11, color="0.25",
                bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.75"))

    def plot_alignment_bars(labels, scores, title, target=0.96):
        gate_45 = 1 / np.sqrt(2)
        fig, ax = plt.subplots(figsize=(10, 4.8))
        bars = ax.bar(labels, scores)
        ax.axhline(target, linestyle="--", linewidth=2, label="max-CGCS target 0.96")
        ax.axhline(gate_45, linestyle=":", linewidth=2, label=r"45° gate $1/\sqrt{1^2+1^2}$")
        for bar, score in zip(bars, scores):
            ax.text(bar.get_x()+bar.get_width()/2, score+0.015, f"{score:.3f}",
                    ha="center", va="bottom", fontsize=11)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel("local CGCS")
        ax.set_title(title, fontsize=18, pad=14)
        ax.legend(loc="lower right")
        return fig, ax

set_style()
FIG_DIR = ensure_fig_dir()
print(f"Figure directory: {FIG_DIR}")

## 1. Hardy-type inequality as variation cost

A model Hardy-type inequality is:

\[
\int_{\mathbb{R}^n}\frac{|u(x)|^2}{|x|^2}\,dx
\le
C\int_{\mathbb{R}^n}|\nabla u(x)|^2\,dx.
\]

Entry-level translation:

> **Concentration must pay variation cost.**

CGCS bridge language:

- **concentration** = localized structure
- **variation cost** = derivative / gradient pressure
- **extend (step)** = structure continues one pipeline step
- **resist (collapse)** = structure survives constraints without contradiction


In [ ]:
x = np.linspace(-5, 5, 2500)

# Wide vs concentrated candidate descriptions.
u_wide = gaussian(x, width=1.25, amplitude=1.0)
u_conc = gaussian(x, width=0.38, amplitude=1.0)

grad_wide = gradient(u_wide, x)
grad_conc = gradient(u_conc, x)

wide_measures = hardy_pair(u_wide, x)
conc_measures = hardy_pair(u_conc, x)

print("Hardy-style finite-grid measures")
print("wide profile:", {k: round(v, 4) for k, v in wide_measures.items()})
print("concentrated profile:", {k: round(v, 4) for k, v in conc_measures.items()})

## 2. Figure — Step 3: Hardy variation cost

Paper-ready filename:

```text
figures/step3_hardy_variation_cost_code.png
```

Recommended caption:

> Step 3. Hardy-type variation cost: concentration must pay variation cost. Highly localized non-zero profiles incur increased gradient cost, preventing simultaneous satisfaction of all constraints.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.2))

ax.plot(x, normalize(u_wide), linewidth=2.5, label="wide profile")
ax.plot(x, normalize(u_conc), linewidth=2.5, label="concentrated profile")
ax.plot(x, normalize(np.abs(grad_wide)), linewidth=2, linestyle="--", label="|gradient| wide")
ax.plot(x, normalize(np.abs(grad_conc)), linewidth=2, linestyle="--", label="|gradient| concentrated")

ax.annotate(
    "concentration",
    xy=(0.0, 1.0),
    xytext=(1.05, 0.92),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=12,
    ha="left",
)

ax.annotate(
    "variation cost",
    xy=(0.38, normalize(np.abs(grad_conc))[np.argmin(np.abs(x - 0.38))]),
    xytext=(1.55, 0.48),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=12,
    ha="left",
)

ax.set_title("Step 3: Hardy-type variation cost", fontsize=18, pad=14)
ax.set_xlabel("space coordinate x")
ax.set_ylabel("normalized profile / gradient")
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(0, 1.15)
ax.legend(loc="upper right", frameon=True)

add_caption(
    ax,
    "AB: Hardy-type bounds link concentration to variation.  NOW/lift5: concentration must pay variation cost.",
)

plt.tight_layout()
out = FIG_DIR / "step3_hardy_variation_cost_code.png"
save(fig, out)
plt.show()

print(f"Saved: {out}")

## 3. Width sweep: concentration cost vs variation cost

A sweep over Gaussian width makes the Hardy intuition computable:

\[
\text{localization increases}
\Rightarrow
\text{variation cost increases}.
\]

This is a finite-grid demonstration, not a formal proof.


In [ ]:
widths = np.linspace(0.25, 2.5, 80)
sweep = width_sweep(x, widths)

concentration_norm = normalize(sweep["concentration"])
variation_norm = normalize(sweep["variation_cost"])

fig, ax = plt.subplots(figsize=(10.8, 5.2))

ax.plot(sweep["widths"], concentration_norm, linewidth=2.5, label="concentration proxy")
ax.plot(sweep["widths"], variation_norm, linewidth=2.5, label="variation-cost proxy")

ax.invert_xaxis()
ax.set_title("Hardy intuition: narrowing width raises constraint pressure", fontsize=18, pad=14)
ax.set_xlabel("Gaussian width (left = more concentrated)")
ax.set_ylabel("normalized proxy")
ax.legend(loc="upper right", frameon=True)

ax.annotate(
    "more concentrated\nprofiles",
    xy=(0.35, concentration_norm[np.argmin(np.abs(sweep["widths"] - 0.35))]),
    xytext=(0.95, 0.82),
    arrowprops=dict(arrowstyle="->", lw=1.4),
    fontsize=11,
    ha="left",
)

add_caption(
    ax,
    "As width decreases, concentration and variation pressure increase together. This is the variation-cost intuition.",
)

plt.tight_layout()
out = FIG_DIR / "notebook02_hardy_width_sweep_code.png"
save(fig, out)
plt.show()

print(f"Saved: {out}")

## 4. Hardy step in the full pipeline

Notebook 01 established:

\[
\text{two-time decay} \rightarrow \text{weighted scale}.
\]

Notebook 02 adds:

\[
\text{concentration} \rightarrow \text{variation cost}.
\]

Together:

```text
decay + weight + Hardy
→ strong constraints on any non-zero solution
```

This prepares Notebook 03:

```text
non-zero solutions satisfy individual constraints, but not all simultaneously
→ only the zero solution remains under all constraints
```


## 5. Local CGCS alignment for Notebook 02

This checks whether the Hardy explanation remains aligned with the standard proof role.

Target:

\[
\text{CGCS} \ge 0.96.
\]


In [ ]:
checks = [
    "Hardy inequality stated",
    "variation cost modeled",
    "AB ↔ NOW language",
]
scores = np.array([0.980, 0.968, 0.965])

fig, ax = plot_alignment_bars(
    labels=checks,
    scores=scores,
    title="Notebook 02: Hardy AB ↔ NOW alignment",
    target=0.96,
)
plt.xticks(rotation=8, ha="right")

add_caption(ax, "Hardy explanation stays above the max-CGCS target.", y=-0.30)

plt.tight_layout()
out = FIG_DIR / "notebook02_hardy_alignment_code.png"
save(fig, out)
plt.show()

print(f"Saved: {out}")

## 6. Export summary

Generated files:

```text
figures/step3_hardy_variation_cost_code.png
figures/notebook02_hardy_width_sweep_code.png
figures/notebook02_hardy_alignment_code.png
```

Next notebook:

- `03_contradiction_pipeline_pde_aligned.ipynb`


In [ ]:
for file in [
    "step3_hardy_variation_cost_code.png",
    "notebook02_hardy_width_sweep_code.png",
    "notebook02_hardy_alignment_code.png",
]:
    p = FIG_DIR / file
    print(f"{p}  exists={p.exists()}")